# Can we predict whether a student passes the closing quiz?
Protocol: `protocol.yaml`. Target `passed_quiz`; group holdout and 5-fold GroupKFold on `student_key`; model chosen by mean CV ROC AUC; holdout scored once.

Features: `enrol_year`, `school_code` (categorical in one candidate), `prior_score` (placement test, before the session), `session_minutes`, `hints_used` (both recorded during the session, before the closing quiz result). `student_key` is excluded (entity key).

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.base import clone

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='5a0b0221ed641db40322e1a95cae32cc67d3525ecb500813f95d6c109b2a7409'
df=pd.read_csv('data.csv')
hold_idx=pd.read_csv('holdout.csv')['row'].to_numpy()
TARGET='passed_quiz'; GROUP='student_key'
FEATURES=['enrol_year','school_code','prior_score','session_minutes','hints_used']
is_hold=np.zeros(len(df),bool); is_hold[hold_idx]=True
train=df[~is_hold]; hold=df[is_hold]
assert not set(train[GROUP])&set(hold[GROUP])
print(len(train),len(hold))

## Candidates (named before any scoring)

In [ ]:
num=['enrol_year','prior_score','session_minutes','hints_used']
candidates={
 'baseline_prior':DummyClassifier(strategy='prior'),
 'logreg':make_pipeline(ColumnTransformer([('num',StandardScaler(),num)]),LogisticRegression(max_iter=1000)),
 'logreg_school_onehot':make_pipeline(ColumnTransformer([('num',StandardScaler(),num),
     ('school',OneHotEncoder(handle_unknown='ignore'),['school_code'])]),LogisticRegression(max_iter=1000)),
 'random_forest':RandomForestClassifier(n_estimators=300,min_samples_leaf=20,random_state=0,n_jobs=-1),
 'hist_gb':HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0),
}
list(candidates)

## Cross-validation on training rows (GroupKFold by student)

In [ ]:
gkf=GroupKFold(n_splits=5)
X=train[FEATURES]; y=train[TARGET].to_numpy(); g=train[GROUP]
train_pos=np.flatnonzero(~is_hold)
folds=list(gkf.split(X,y,g))
cv_scores={}
for name,m in candidates.items():
    s=[]
    for tr,va in folds:
        mm=clone(m).fit(X.iloc[tr],y[tr])
        s.append(roc_auc_score(y[va],mm.predict_proba(X.iloc[va])[:,1]))
    cv_scores[name]=s
cv=pd.DataFrame({k:{'mean':np.mean(v),'std':np.std(v)} for k,v in cv_scores.items()}).T
cv.sort_values('mean',ascending=False)

## Choose by mean CV ROC AUC, refit on all training rows, score holdout once

In [ ]:
chosen=max(cv_scores,key=lambda k:np.mean(cv_scores[k]))
final=clone(candidates[chosen]).fit(X,y)
holdout_auc=roc_auc_score(hold[TARGET],final.predict_proba(hold[FEATURES])[:,1])
print(chosen, 'CV mean',np.mean(cv_scores[chosen]),'holdout ROC AUC',holdout_auc)

In [ ]:
evidence={
 'features':FEATURES,
 'candidates':list(candidates),
 'cv_folds':[[train_pos[tr].tolist(),train_pos[va].tolist()] for tr,va in folds],
 'cv_scores':{k:[float(x) for x in v] for k,v in cv_scores.items()},
 'chosen':chosen,
 'holdout_rows':hold_idx.tolist(),
 'holdout_scores':{chosen:float(holdout_auc)},
}
json.dump(evidence,open('evidence.json','w'))

## Conclusion
See the CV table and holdout ROC AUC above. The holdout score is the headline; it was computed once, after the choice was made on CV.